In [8]:
import numpy as np 
import pandas as pd 
import os

# --- 1. EXTRACT: Dynamic Path Discovery ---
def get_data_path():
    """Finds the CSV file regardless of the exact filename."""
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            if filename.endswith('.csv'):
                return os.path.join(dirname, filename)
    return None

# --- 2. TRANSFORM: The Cleaning Engine ---
def run_cleaning_pipeline(df):
    """
    Standardizes column names, handles missing values, 
    and performs feature engineering.
    """
    # Standardize column names (lowercase and underscores)
    df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]
    
    # Identify emission columns (Scope 1, 2, 3)
    # Using 'contains' makes it resilient to naming variations
    emission_cols = [c for c in df.columns if 'scope' in c]
    
    for col in emission_cols:
        # Convert to numeric, turning errors (like 'N/A' or strings) into NaN
        df[col] = pd.to_numeric(df[col], errors='coerce')
        # Fill NaNs with 0 - a common engineering assumption for missing reports
        df[col] = df[col].fillna(0)
    
    # Feature Engineering: Total Carbon Footprint
    df['total_ghg_emissions'] = df[emission_cols].sum(axis=1)
    
    # Engineering Metric: Emissions relative to Revenue/EBITDA (if available)
    if 'ebitda' in df.columns:
        df['ebitda'] = pd.to_numeric(df['ebitda'], errors='coerce').fillna(0)
        # Avoid division by zero
        df['carbon_intensity'] = df['total_ghg_emissions'] / df['ebitda'].replace(0, np.nan)
        
    return df

# --- 3. LOAD: Save for Analysis ---
def save_processed_data(df, name="processed_emissions.parquet"):
    """Saves to Parquet for 4GB RAM efficiency."""
    df.to_parquet(name)
    print(f"✅ Pipeline Complete. Processed data saved as: {name}")

# --- EXECUTION BLOCK ---
data_path = get_data_path()

if data_path:
    print(f"📂 Found dataset at: {data_path}")
    raw_df = pd.read_csv(data_path)
    
    # Run the pipeline
    clean_df = run_cleaning_pipeline(raw_df)
    
    # Save the output
    save_processed_data(clean_df)
    
    # Peek at the results
    print(clean_df.head())
else:
    print("❌ No CSV found. Please check your Kaggle 'input' folder.")

📂 Found dataset at: /kaggle/input/datasets/alitaqishah/global-corporate-ghg-emissions-20222023/global_corporate_ghg_emissions_2022_2023.csv
✅ Pipeline Complete. Processed data saved as: processed_emissions.parquet
  company_id     company_name ticker       country        region     sector  \
0    GHG0001            Sasol    SOL  South Africa        Africa  Materials   
1    GHG0002            Eskom      —  South Africa        Africa  Utilities   
2    GHG0003  Woodside Energy    WDS     Australia  Asia-Pacific     Energy   
3    GHG0004           Santos    STO     Australia  Asia-Pacific     Energy   
4    GHG0005              BHP    BHP     Australia  Asia-Pacific  Materials   

         industry  reporting_year  scope1_mt_co2e  scope2_location_mt_co2e  \
0       Chemicals            2022            47.3                      2.1   
1  Electric Power            2022           195.0                      0.0   
2       E&P (LNG)            2022             7.1                      0.5   

In [9]:
def validate_data(df):
    """
    Performs quality checks on the processed data.
    Prints a health report for the engineering logs.
    """
    print("--- 🩺 DATA HEALTH REPORT ---")
    
    # Check 1: Negative Values
    neg_emissions = df[df['total_ghg_emissions'] < 0].shape[0]
    print(f"❌ Negative Emission Records: {neg_emissions}")
    
    # Check 2: Missing Revenue (Crucial for Intensity Analysis)
    missing_rev = df['revenue_usd_millions'].isna().sum()
    rev_pct = (missing_rev / len(df)) * 100
    print(f"⚠️ Missing Revenue: {missing_rev} ({rev_pct:.2f}%)")
    
    # Check 3: Logical Verification (Scope 1+2 should usually be <= Total)
    # If this fails, our sum logic or the source data is skewed
    logic_fail = df[df['scope1_mt_co2e'] + df['scope2_location_mt_co2e'] > df['total_ghg_emissions']].shape[0]
    print(f"🔍 Logic Fails (S1+S2 > Total): {logic_fail}")
    
    # Check 4: The "Whale" Polluters (Top 1%)
    threshold = df['total_ghg_emissions'].quantile(0.99)
    whales = df[df['total_ghg_emissions'] > threshold]
    print(f"🐳 Top 1% Polluter Threshold: {threshold:.2f} MT CO2e")
    
    return whales # Return the outliers for manual inspection

# EXECUTION
top_polluters = validate_data(clean_df)
print("\n--- TOP 1% POLLUTERS PREVIEW ---")
print(top_polluters[['company_name', 'sector', 'total_ghg_emissions']].head())

--- 🩺 DATA HEALTH REPORT ---
❌ Negative Emission Records: 0
⚠️ Missing Revenue: 64 (12.60%)
🔍 Logic Fails (S1+S2 > Total): 0
🐳 Top 1% Polluter Threshold: 469.85 MT CO2e

--- TOP 1% POLLUTERS PREVIEW ---
             company_name     sector  total_ghg_emissions
16   China Shenhua Energy     Energy                640.4
42    China Huaneng Group  Utilities                800.0
178         TotalEnergies     Energy                471.0
240               Gazprom     Energy               1436.4
277                 Shell     Energy                729.4


In [10]:
def engineer_financial_features(df):
    """
    Imputes missing revenue based on sector medians 
    to preserve data for intensity analysis.
    """
    print("--- 🛠️ REVENUE IMPUTATION START ---")
    
    # 1. Calculate median revenue per sector
    sector_medians = df.groupby('sector')['revenue_usd_millions'].transform('median')
    
    # 2. Fill missing revenue with the sector-specific median
    before_count = df['revenue_usd_millions'].isna().sum()
    df['revenue_usd_millions'] = df['revenue_usd_millions'].fillna(sector_medians)
    after_count = df['revenue_usd_millions'].isna().sum()
    
    print(f"✅ Successfully imputed {before_count - after_count} revenue records.")
    
    # 3. Recalculate Emissions Intensity now that we have more revenue data
    # Intensity = Total Emissions / Revenue
    df['carbon_intensity'] = df['total_ghg_emissions'] / df['revenue_usd_millions']
    
    return df

# EXECUTION
final_df = engineer_financial_features(clean_df)

# Check a company that was missing revenue before (like Eskom at index 1)
print("\n--- VERIFYING IMPUTATION (Index 1: Eskom) ---")
print(final_df.loc[[1], ['company_name', 'sector', 'revenue_usd_millions', 'carbon_intensity']])

--- 🛠️ REVENUE IMPUTATION START ---
✅ Successfully imputed 64 revenue records.

--- VERIFYING IMPUTATION (Index 1: Eskom) ---
  company_name     sector  revenue_usd_millions  carbon_intensity
1        Eskom  Utilities                7999.5          0.048753


In [11]:
def scale_features(df):
    """
    Normalizes emissions and revenue to a 0-1 range.
    Essential for future ML models (Clustering, Regression).
    """
    print("--- ⚖️ FEATURE SCALING START ---")
    
    cols_to_scale = ['total_ghg_emissions', 'revenue_usd_millions', 'carbon_intensity']
    
    for col in cols_to_scale:
        min_val = df[col].min()
        max_val = df[col].max()
        
        # Min-Max Scaling Formula: (x - min) / (max - min)
        df[f'{col}_scaled'] = (df[col] - min_val) / (max_val - min_val)
        
    print(f"✅ Scaled {len(cols_to_scale)} features to [0, 1] range.")
    return df

# EXECUTION
final_df = scale_features(final_df)

# Final Peek at the Master Dataset
print("\n--- MASTER DATASET PREVIEW (SCALED) ---")
print(final_df[['company_name', 'total_ghg_emissions_scaled', 'carbon_intensity_scaled']].head())

--- ⚖️ FEATURE SCALING START ---
✅ Scaled 3 features to [0, 1] range.

--- MASTER DATASET PREVIEW (SCALED) ---
      company_name  total_ghg_emissions_scaled  carbon_intensity_scaled
0            Sasol                    0.068679                 0.167674
1            Eskom                    0.271431                 0.487482
2  Woodside Energy                    0.010472                 0.009004
3           Santos                    0.009219                 0.018424
4              BHP                    0.025372                 0.005587


In [14]:
import plotly.express as px

def create_sector_dashboard(df):
    """
    Groups data by sector and creates a Tableau-style interactive bar chart.
    """
    # 1. Aggregate data for the visualization
    # We group by sector and take the mean of our new 'carbon_intensity' feature
    analysis = df.groupby('sector')['carbon_intensity'].mean().sort_values(ascending=False).reset_index()
    
    # 2. Build the Plotly figure
    fig = px.bar(analysis, 
                 x='carbon_intensity', 
                 y='sector', 
                 orientation='h',
                 title='Global Carbon Intensity by Sector (Interactive)',
                 labels={'carbon_intensity': 'Avg Emissions per $1M Revenue', 'sector': 'Industry Sector'},
                 color='carbon_intensity',
                 color_continuous_scale='Viridis',
                 template='plotly_dark') # High-end dashboard look

    # 3. Refine layout for professional feel
    fig.update_layout(
        xaxis_tickformat='.4f',
        yaxis={'categoryorder':'total ascending'},
        hovermode="closest"
    )
    
    return fig, analysis

# EXECUTION
fig_sector, sector_analysis = create_sector_dashboard(final_df)
fig_sector.show()

In [15]:
def create_company_drilldown(df):
    """
    Creates an interactive scatter plot allowing for per-company inspection.
    """
    fig = px.scatter(df, 
                     x='revenue_usd_millions', 
                     y='total_ghg_emissions',
                     color='sector',
                     size='carbon_intensity_scaled', # Bigger bubbles = less efficient
                     hover_name='company_name', 
                     log_x=True, 
                     log_y=True,
                     title='Corporate Emissions vs Revenue (Interactive Drill-down)',
                     labels={'revenue_usd_millions': 'Revenue (Log Scale $M)', 
                             'total_ghg_emissions': 'Total Emissions (Log Scale MT)'},
                     template='plotly_white')
    
    return fig

# EXECUTION
fig_drilldown = create_company_drilldown(final_df)
fig_drilldown.show()